# Slotfill - phase 4: evaluate the tuned adapter on the held-out sets

Serves the base model plus the LoRA adapter with vLLM **inside this session** (nothing is
deployed, nothing is exposed) and runs `eval/predict.py` against it, through the same harness
and the same scoring the frontier baseline went through. The only difference is the transport.

The adapter is evaluated on the prompt it was trained on: page image + one sentence,
`--no-example --no-schema`. The baseline got the schema and a worked example; that asymmetry
is the experiment, not a mistake.

**What it costs:** free-tier T4 time. ~15 min of setup, then ~1,083 pages through vLLM.
**Risk to watch:** vLLM on a T4 (compute capability 7.5, no bfloat16) is the one thing that
cannot be verified off the GPU. Cell 6 sends a single request before the long run for exactly
that reason - if it fails, stop and report the error rather than burning quota.

**Prerequisite:** the trained adapter on Drive, from `train/colab_train.ipynb`.

In [ ]:
PLATFORM = 'colab'   # or 'kaggle'

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content'
    ADAPTER = '/content/drive/MyDrive/slotfill/adapters/qwen25vl3b-r16-train/adapter'
elif PLATFORM == 'kaggle':
    ROOT = '/kaggle/working'
    ADAPTER = '/kaggle/input/slotfill-adapter/adapter'
else:
    raise ValueError(PLATFORM)

import os
assert os.path.exists(f'{ADAPTER}/adapter_config.json'), ADAPTER
print('adapter:', ADAPTER)

In [ ]:
%%capture
!pip install vllm

In [ ]:
import subprocess

REPO = 'https://github.com/InaPD/invoiception.git'
BRANCH = 'eval-adapter-phase4'   # or master once merged
repo_dir = f'{ROOT}/invoiception'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, repo_dir], check=True)
%cd {repo_dir}
!git pull --ff-only
!pip install -q -e . --no-deps && pip install -q jsonschema pillow openai
!git log --oneline -1

## The held-out data

The frozen sets were deliberately left out of the training bundle, so they are fetched from
Zenodo here (~730 MB, md5-verified by the same script that fetched them locally). The split
manifests are committed in the repo, so the exact same documents are scored as before.

In [ ]:
!python -m data.download
!python -c "from eval.datasets import load_eval_set; \
  print({s: len(load_eval_set(s)) for s in ('test_seen','test_unseen','rvlcdip')})"

## Start vLLM

Base model + adapter behind one endpoint; the OpenAI `model` field picks which. First start
downloads the base model (~7.5 GB) and takes several minutes. The server runs in the
background; the next cell waits for it.

In [ ]:
import subprocess, time

log = open(f'{ROOT}/vllm.log', 'w')
server = subprocess.Popen(['bash', 'serve/serve_vllm.sh', ADAPTER], stdout=log, stderr=log)
print('starting, pid', server.pid)

In [ ]:
import urllib.request, json, time

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f'vLLM exited with {server.returncode}; see {ROOT}/vllm.log')
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5) as r:
            print([m['id'] for m in json.load(r)['data']])
            break
    except Exception:
        time.sleep(10)
else:
    raise TimeoutError('vLLM did not come up in 15 minutes')

## One request first

Two pages from `dev_unseen` (not frozen, so this costs nothing that matters) through the real
code path. If this prints a valid record, the long run is safe to start.

In [ ]:
!python -m eval.predict --set dev_unseen --input image --limit 2 \
    --backend openai --base-url http://localhost:8000/v1 --model slotfill-lora \
    --condition adapter-check --no-example --no-schema --max-tokens 1024 --fresh
!python -m eval.evaluate runs/adapter-check/dev_unseen

## The held-out run

`--held-out` is required on purpose: these three sets have never been iterated on. Roughly
1,083 pages. Runs resume, so a dropped session costs only what it had not finished.

In [ ]:
%%time
!python -m eval.predict --set test_seen --input image \
    --backend openai --base-url http://localhost:8000/v1 --model slotfill-lora \
    --condition vision-adapter --no-example --no-schema --max-tokens 1024 \
    --concurrency 16 --held-out

In [ ]:
%%time
!python -m eval.predict --set test_unseen --input image \
    --backend openai --base-url http://localhost:8000/v1 --model slotfill-lora \
    --condition vision-adapter --no-example --no-schema --max-tokens 1024 \
    --concurrency 16 --held-out

In [ ]:
%%time
!python -m eval.predict --set rvlcdip --input image \
    --backend openai --base-url http://localhost:8000/v1 --model slotfill-lora \
    --condition vision-adapter --no-example --no-schema --max-tokens 1024 \
    --concurrency 16 --held-out

## Score it

`--gpu-usd-per-hour` turns the measured throughput into the cost column. 0.35 is a rough
market rate for a T4; pass whatever the GPU actually costs (0 is not the answer - free to
you is not free to run).

In [ ]:
!python -m eval.evaluate --gpu-usd-per-hour 0.35 \
    runs/vision-adapter/test_seen runs/vision-adapter/test_unseen runs/vision-adapter/rvlcdip

## Bring the results home

`runs/vision-adapter/<set>/` holds `run_config.json`, `predictions.jsonl`, `sessions.jsonl`
and `metrics.json`. Download the folder and commit it: these are the ship-gate numbers.

Then stop the server and the runtime; idle GPU time still counts against the quota.

In [ ]:
!cd runs && tar -czf {ROOT}/vision-adapter-runs.tar.gz vision-adapter && ls -la {ROOT}/vision-adapter-runs.tar.gz
from google.colab import files   # Kaggle: the file is already in /kaggle/working
files.download(f'{ROOT}/vision-adapter-runs.tar.gz')

In [ ]:
server.terminate()
print('server stopped; now Runtime -> Disconnect and delete runtime')

### If vLLM will not run on this GPU

A T4 is old (2018, compute capability 7.5, no bfloat16) and vLLM's newer kernels increasingly
assume 8.0+. If cell 7 or 8 fails, read `vllm.log`, and:

- try `DTYPE=float16` variations or an older vLLM (`pip install vllm==0.11.0`) first;
- otherwise switch this notebook to a paid L4 runtime (~$1-2 for the whole run, inside the
  plan's budget line), which has bfloat16 and is the boring, working path;
- Kaggle's free GPUs are T4 and P100, so they are not a way around this.

Report the error before spending money: the fix may be a flag.